# 3. Association Rules (Leena)

- **Goal:** Pricing Patterns & Marketing Insights.
- **Implementation:** Use Apriori or FP-Growth algorithms on categorical features (`BRAND_TIER`, `CPU_TIER`, `STORAGE_TYPE`).
- **Insight:** Discover rules like `{GPU=RTX, RAM=16GB} => {Price=Premium}`. This explains the market logic.

## 📊 Evaluation & Validation (70/30 Split)
This notebook uses the training dataset (70 split) to discover market patterns.

In [ ]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Configuration
MIN_SUPPORT = 0.05       # Minimum percentage of transactions containing the itemset
MIN_CONFIDENCE = 0.5     # Minimum probability of the consequent given the antecedent
TARGET_FEATURES = ['BRAND_TIER', 'CPU_TIER', 'STORAGE_TYPE', 'RAM_SIZE'] 
# Added RAM_SIZE as it's a key spec driver alongside the requested ones

### 1. Load Data

In [ ]:
try:
    df = pd.read_csv('training_dataset.csv')
except FileNotFoundError:
    # Fallback for running from different relative paths
    df = pd.read_csv('06_Model_Training_Evaluation/training_dataset.csv')

print(f"Loaded dataset with shape: {df.shape}")
df.head()

### 2. Preprocessing
- Select Categorical Features
- Bin `PRICE` into Tiers (`Budget`, `Mainstream`, `Premium`, `High-End`)
- One-Hot Encode data for the algorithm

In [ ]:
# Filter for selected columns
data = df[TARGET_FEATURES + ['PRICE']].copy()

# Create Price Tiers
# We use qcut to ensure roughly equal distribution of classes, or cut for fixed ranges
# Using qcut (Quantitative Cut) for quartiles
try:
    data['Price_Tier'] = pd.qcut(data['PRICE'], q=4, labels=['Budget', 'Mainstream', 'Premium', 'High-End'])
except ValueError:
    # Fallback if specific bin edges are needed or too many duplicate edges
    data['Price_Tier'] = pd.cut(data['PRICE'], bins=4, labels=['Budget', 'Mainstream', 'Premium', 'High-End'])

# Drop numerical Price
data = data.drop(columns=['PRICE'])

# Handle missing
data.dropna(inplace=True)

# Convert all to string to ensure they are treated as specific categories
data = data.astype(str)

print("Data for Mining:")
print(data.head())

In [ ]:
# One-Hot Encoding
data_encoded = pd.get_dummies(data)
data_encoded = data_encoded.astype(bool)

print(f"Encoded shape: {data_encoded.shape}")

### 3. Mining Frequent Itemsets (FP-Growth)

In [ ]:
frequent_itemsets = fpgrowth(data_encoded, min_support=MIN_SUPPORT, use_colnames=True)

print(f"Found {len(frequent_itemsets)} frequent itemsets.")
frequent_itemsets.sort_values(by='support', ascending=False).head()

### 4. Generating Association Rules
We are looking for strong implications, so we filter by `Confidence` and sort by `Lift`.

In [ ]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)

print(f"Generated {len(rules)} rules.")

### 5. Evaluation: Pricing Patterns
Filtering rules where the **Consequent** (Right-hand side) is a **Price Tier**.

In [ ]:
# Filter for rules leading to a Price Tier
def is_price_tier(s):
    return any("Price_Tier" in item for item in s)

price_rules = rules[rules['consequents'].apply(is_price_tier)].copy()

# Sort by Lift
price_rules = price_rules.sort_values(by='lift', ascending=False)

# Formatting helper
def format_frozenset(s):
    return ', '.join(list(s))

price_rules['antecedents_str'] = price_rules['antecedents'].apply(format_frozenset)
price_rules['consequents_str'] = price_rules['consequents'].apply(format_frozenset)

display_cols = ['antecedents_str', 'consequents_str', 'confidence', 'lift', 'support']

# Display Top Rules
print("Top Pricing Rules (Confidence > 50%):")
price_rules[display_cols].head(20)

## 6. Visualization of Result Table
Outputting the final table as requested.

In [ ]:
print(f"{'Association Rules':<60} | {'Confidence':<10} | {'Lift':<10}")
print("-" * 90)

for idx, row in price_rules.head(15).iterrows():
    rule = f"{{{row['antecedents_str']}}} => {{{row['consequents_str']}}}"
    conf = f"{row['confidence']:.2f}"
    lift = f"{row['lift']:.2f}"
    print(f"{rule:<60} | {conf:<10} | {lift:<10}")